In [1]:
import numpy as np
from scipy import signal as sig
import matplotlib.pyplot as plt
import Funciones as Funciones
from NNMF_GIF import NNMF_GIF 
from pathlib import Path
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from IPython.display import  clear_output
import pandas as pd
#%config InlineBackend.figure_format = 'svg'
import librosa as lib

In [2]:
def IF_SNF_OneMetric(x, fs, numMetric, f0=None):
    """
    Inverse filtering using a Single Notch Filter optimized by one metric.

    Parameters:
        x         : array-like, input signal
        fs        : float, sampling frequency (Hz)
        numMetric : int, metric index to optimize (1, 2 or 3)
        f0        : float (optional), starting frequency for search (Hz)
                    if None, search starts at 300 Hz

    Returns:
        gvv : inverse filtered signal
        br  : numerator filter coefficient
        ar  : denominator filter coefficients
        F1  : estimated first formant frequency (Hz)
        BW1 : estimated first formant bandwidth (Hz)
    """
    x0 = np.array(x, dtype=float).flatten()

    if f0 is not None:
        fAR = np.arange(f0, 960, 10)   # f0:10:950
    else:
        fAR = np.arange(300, 960, 10)  # 300:10:950

    BW = np.array([70.0])

    n_fAR = len(fAR)
    n_BW  = len(BW)

    metric_01     = np.full((n_fAR, n_BW), np.nan)
    metric_01min  = np.full(n_fAR, np.nan)
    metric_01_I   = np.zeros(n_fAR, dtype=int)

    for ii, fPole in enumerate(fAR):
        for jj, BWpole in enumerate(BW):
            x1, _, _ = SNF(x0, fPole, fs, BWpole)
            m = metricsIF(x1, x0)
            metric_01[ii, jj] = m[numMetric - 1]  # 1-based to 0-based

        metric_01min[ii] = np.min(metric_01[ii, :])
        metric_01_I[ii]  = np.argmin(metric_01[ii, :])

    nMinfAR1 = np.argmin(metric_01min)
    nMinBW   = metric_01_I[nMinfAR1]

    F1  = fAR[nMinfAR1]
    BW1 = BW[nMinBW]

    gvv, br, ar = SNF(x0, F1, fs, BW1)
    gvv = gvv.flatten()

    return gvv, br, ar, F1, BW1

def _tukeywin(n, r=0.5):
    """Tukey (tapered cosine) window — equivalent to MATLAB's tukeywin."""
    if r <= 0:
        return np.ones(n)
    if r >= 1:
        return np.hanning(n)
    w    = np.ones(n)
    half = int(np.floor(r * n / 2))
    for i in range(half):
        w[i] = 0.5 * (1 - np.cos(2 * np.pi * i / (r * n)))
    for i in range(n - half, n):
        w[i] = w[n - 1 - i]
    return w


def metricsIF(x, x0):
    """
    Compute inverse filtering quality metrics 1, 2 and 3.

    Parameters:
        x  : array-like, inverse filtered (IF) signal
        x0 : array-like, original (non-filtered) signal

    Returns:
        metrics : array of 3 values [metric1, metric2, metric3]
    """
    x  = np.array(x,  dtype=float).flatten()
    x0 = np.array(x0, dtype=float).flatten()

    # Tukey window
    w  = _tukeywin(len(x), 0.5)
    x  = x  * w
    x0 = x0 * w

    lenx = len(x)
    x1   = x.copy()

    # Differences
    x0diff     = np.diff(x0)
    x1diff     = np.diff(x1)
    x0diffdiff = sig.filtfilt([1, -1], [1], x0)
    x1diffdiff = sig.filtfilt([1, -1], [1], x1)

    # Central segment (25% to 75%)
    i0 = int(np.floor(lenx * 0.25))-1
    i1 = int(np.floor(lenx * 0.75))

    x1         = x1[i0:i1]
    x0         = x0[i0:i1]
    x1diff     = x1diff[i0:i1]
    x0diff     = x0diff[i0:i1]
    x1diffdiff = x1diffdiff[i0:i1]
    x0diffdiff = x0diffdiff[i0:i1]

    metrics = np.full(3, np.nan)

    # Metric 1: sum(|Δ²x|)
    metrics[0] = np.sum(np.abs(x1diffdiff)) / np.sum(np.abs(x0diffdiff))

    # Metric 2: F99(x) - F1(x)
    metrics[1] = (np.percentile(x1, 99) - np.percentile(x1, 1)) / \
                 (np.percentile(x0, 99) - np.percentile(x0, 1))

    # Metric 3: sum(|Δx|)
    metrics[2] = np.sum(np.abs(x1diff)) / np.sum(np.abs(x0diff))

    return metrics


def vtractz(p, bw, fs):
    """
    Produce a vector of z-domain poles corresponding to input pole frequencies
    and bandwidths, assuming an all-pole model of the vocal tract.

    Parameters:
        p  : array-like, pole frequencies in Hz
        bw : array-like, pole bandwidths in Hz
        fs : float, sampling frequency in Hz

    Returns:
        zp : numpy array of complex z-domain poles (pairs a+jb, a-jb)
    """
    p  = np.atleast_1d(np.array(p, dtype=float))
    bw = np.atleast_1d(np.array(bw, dtype=float))

    n  = len(p)
    zp = np.zeros(2 * n, dtype=complex)

    B = np.pi * bw / fs          # Bandwidths in radians
    P = 2 * np.pi * p / fs       # Pole frequencies in radians

    # Z-plane pole magnitudes
    r = -np.cos(B) + 2 - np.sqrt((np.cos(B) - 1) * (np.cos(B) - 3))

    for m in range(n):
        if p[m] < fs / 4:        # 1st quadrant (positive real part)
            a = r[m] / np.sqrt(1 + np.tan(P[m])**2)
        else:                    # 2nd quadrant (negative real part)
            a = -r[m] / np.sqrt(1 + np.tan(P[m])**2)

        b = a * np.tan(P[m])     # Imaginary part

        zp[2 * m]     = complex(a,  b)
        zp[2 * m + 1] = complex(a, -b)

    return zp


def SNF(x, fc, fs, BW=None):
    """
    Single Notch Filter with unitary gain at DC.

    Parameters:
        x  : array-like, input signal
        fc : array-like, notch frequency/ies in Hz
        fs : float, sampling frequency in Hz
        BW : array-like (optional), bandwidths in Hz (default: 70 Hz for each fc)

    Returns:
        x_filt : filtered signal
        b      : numerator filter coefficients
        a      : denominator filter coefficients
    """
    fc = np.atleast_1d(np.array(fc, dtype=float))
    x  = np.array(x, dtype=float)

    if BW is None:
        BW = 70 * np.ones(len(fc))
    else:
        BW = np.atleast_1d(np.array(BW, dtype=float))

    zp = vtractz(fc, BW, fs)       # Z-domain poles

    # zp2tf with no zeros, poles=zp, gain=1
    # scipy's zpk2tf returns (b, a) → here 'a' is the denominator
    _, a = sig.zpk2tf([], zp, 1)

    # Gain normalization: b = sum(a) ensures unity gain at DC (z=1)
    b = np.array([np.sum(a)])

    x_filt = sig.lfilter(a,b, x)

    return x_filt, b, a

def IF_OnePole_fC_BW(x, fs):
    """
    Inverse filtering of bandlimited (<1 kHz) signals.

    Parameters:
        x  : array-like, input signal (LP filtered <1kHz, no DC component)
        fs : float, sampling frequency (Hz)

    Returns:
        gvv : inverse filtered signal
        br  : numerator filter coefficient (scalar)
        ar  : denominator filter coefficients (array)
        er  : minimum error metric achieved
        F1  : estimated first formant frequency (Hz)
        BW1 : estimated first formant bandwidth (Hz)
    """
    x = np.array(x, dtype=float)
    x0 = x.copy()
    lenx = len(x)

    fAR = np.arange(200, 1110, 10)   # 200:10:1100
    BW  = np.arange(30,   60,  10)   # 30:10:50

    n_fAR = len(fAR)
    n_BW  = len(BW)

    fPole  = np.zeros((n_fAR, n_BW))
    BWPole = np.zeros((n_fAR, n_BW))
    er     = np.zeros((n_fAR, n_BW))

    # Butterworth post-IF filter (fixed, computed once)
    b_but, a_but = sig.butter(4, 2500 / (fs / 2))

    # Store SNF coefficients
    b1 = np.zeros((n_fAR, n_BW))
    a1 = np.zeros((n_fAR, n_BW, 5))   # cheby2 order-2 → denominator has 5 coeffs (2*n+1)

    i_start = int(np.floor(lenx * 0.25))
    i_end   = int(np.floor(lenx * 0.75))

    for jj, bw_val in enumerate(BW):
        for ii, f_val in enumerate(fAR):

            fPole[ii, jj]  = f_val
            BWPole[ii, jj] = bw_val

            x1, b_snf, a_snf = SNF(x0, f_val, fs, bw_val)

            b1[ii, jj]          = b_snf[0]
            a1[ii, jj, :len(a_snf)] = a_snf

            aux1 = x1[i_start:i_end].copy()

            # Post inverse-filter smoothing
            aux1 = sig.filtfilt(b_but, a_but, aux1)

            metric1    = np.abs(np.diff(np.diff(aux1)))
            er[ii, jj] = np.sum(metric1)

    # Find minimum error indices
    nx      = np.argmin(er, axis=0)          # best fAR index for each BW
    ery     = er[nx, np.arange(n_BW)]        # minimum er per BW column
    ny      = np.argmin(ery)                 # best BW index

    F1  = fPole[nx[ny],  ny]
    BW1 = BWPole[nx[ny], ny]

    br  = np.array([b1[nx[ny], ny]])
    ar  = a1[nx[ny], ny, :]
    ar  = ar[ar != 0] if np.any(ar != 0) else ar   # trim trailing zeros

    gvv, _, _ = SNF(x, F1, fs, BW1)
    er_min    = ery[ny]

    return gvv, br, ar, er_min, F1, BW1


def LP_Filter(x, fs, fc=1100, order=10):
    x = np.array(x, dtype=float)

    # Diseño del filtro Chebyshev tipo II en forma directa (coeficientes b, a)
    b, a = sig.cheby2(order, 40, fc / (fs / 2), btype='low', output='ba')

    # Aplicación del filtro con filtfilt para evitar desfase
    y = sig.filtfilt(b, a, x)

    return y,b,a


def HP_Filter(x, fs, fc=60, order=6):
    """
    High pass filter for voice analysis.

    Parameters:
        x     : array-like, input signal
        fs    : float, sampling frequency (Hz)
        fc    : float, cutoff frequency in Hz (default: 60)
        order : int, filter order (default: 6)

    Returns:
        x : filtered signal
    """
    x = np.array(x, dtype=float)

    b, a = sig.butter(order, fc / (fs / 2), btype='high')
    x    = sig.filtfilt(b, a, x)

    return x,b,a


In [3]:
FC = FileChooser("G:\\Mi unidad\\Facultad\\Beca CIN\\NNMF-GIF\\Señales VPLAB")
FC.layout = widgets.Layout(width='1000px')
display(FC)

FileChooser(path='G:\Mi unidad\Facultad\Beca CIN\NNMF-GIF\Señales VPLAB', filename='', title='', show_hidden=F…

In [11]:
filtro_pb            = 'Chebyshev II'
#filtro_pb           = 'Butterworth'
orden_filtro_pb      = 10
atenuacion_filtro_pb = 40
fc_filtro_pb         = 1100
orden_filtro_pa      = 6
fc_filtro_pa         = 60
metrica = 1

In [14]:
%matplotlib widget

plt.close('all')
clear_output(wait=True)

archivo = Path(FC.selected)
nombre_archivo = archivo.name

df = pd.read_csv(FC.selected, sep=',')

tiempos      = df['t_ms'].values
OVV_original = df['OVV'].values

linewidth   = 2
fs_r        = 8000
color_notch = 'red'

OVV_original                   = OVV_original / max(abs(OVV_original))
OVV_filtrada_pb, b_pb, a_pb    = LP_Filter(OVV_original, fs=fs_r, fc=fc_filtro_pb, order=orden_filtro_pb)
w_pb, h_pb                     = sig.freqz(b_pb, a_pb, fs=fs_r)
OVV_filtrada_pb_pa, b_pa, a_pa = HP_Filter(OVV_filtrada_pb, fs=fs_r)

global FF, BW1, flujo_ovv
flujo_ovv, br, ar, FF, BW1 = IF_SNF_OneMetric(OVV_filtrada_pb_pa, fs=fs_r, numMetric=metrica)

# --- Widgets ---
slider = widgets.IntSlider(
    value=FF, min=200, max=1000, step=1,
    description='Frecuencia [Hz]:',
    style={'description_width': 'initial'},
)
reset_button   = widgets.Button(description="Defecto",
                                layout=widgets.Layout(width='100px'))
guardar_button = widgets.Button(description="Guardar")
orden_filtro   = widgets.BoundedIntText(
    value=orden_filtro_pb, min=2, max=20, step=1,
    description='Orden del filtro pasa bajos:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='210px')
)
fc_filtro = widgets.BoundedIntText(
    value=fc_filtro_pb, min=200, max=2000, step=10,
    description='Frecuencia de corte [Hz]:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

# --- Figura persistente ---
fig, ax = plt.subplots(2, 2, figsize=(13, 5))
fig.canvas.header_visible = False
fig.subplots_adjust(hspace=0.30)
ax0, ax1, ax2, ax3 = ax[0][0], ax[1][0], ax[0][1], ax[1][1]

fig_out = widgets.Output()
with fig_out:
    plt.show()   # ← registra el canvas interactivo dentro del Output

# --- Funciones ---


def _dibujar(OVV_filtrada_pb_pa, w_pb, h_pb, frec_slider):
    global BW1, flujo_OVV

    h_ovv_original, w_ovv_original = Funciones.FFT(OVV_original, fs_r)
    h_pb_pa, w_pb_pa               = Funciones.FFT(OVV_filtrada_pb_pa, fs_r)

    flujo_OVV, b, a = SNF(OVV_filtrada_pb_pa, frec_slider, fs_r, BW1)


    h_flujo_OVV, w_flujo_OVV = Funciones.FFT(flujo_OVV, fs_r)
    w_notch, h_notch          = sig.freqz(a, b, fs=fs_r)

    ax0.clear()
    ax1.clear()
    ax2.clear()
    ax3.clear()
    
    ax0.plot(tiempos, OVV_original, color='blue', label='OVV original', linewidth=linewidth)
    ax0.legend(loc='upper right',fontsize=8)
    ax0.set_xlabel('Tiempo [ms]')
    ax0.set_ylabel('Magnitud')
    ax0.set_title(nombre_archivo.removesuffix('.csv'), pad=20)
    ax0.set_xlim([10, 30])
    to_mascara = 5
    tf_mascara = 50
    mask = (tiempos >= to_mascara) & (tiempos <= tf_mascara)
    flujo_OVV_ = flujo_OVV[mask]
    OVV_filtrada_pb_pa_ = OVV_filtrada_pb_pa[mask]
    tiempos_ = tiempos[mask]
    ax1.plot(tiempos_, OVV_filtrada_pb_pa_, color='k',alpha=0.75,          label='OVV filtrada PA-PB', linewidth=linewidth)
    ax1.plot(tiempos_, flujo_OVV_,          color=color_notch, label='Flujo',              linewidth=linewidth)
    ax1.legend(loc='upper right',fontsize=8)
    ax1.set_xlabel('Tiempo [ms]')
    ax1.set_ylabel('Magnitud')
    ax1.set_xlim([10, 30])

    ax2.plot(w_notch,        10*np.log10(abs(h_notch)),        color=color_notch, label='Notch',        linewidth=linewidth)
    ax2.plot(w_pb,           10*np.log10(abs(h_pb)),           color='green',     label='PB',           linewidth=linewidth)
    ax2.plot(w_ovv_original, 10*np.log10(abs(h_ovv_original)), color='blue',      label='OVV original', linewidth=linewidth)
    ax2.legend(loc='lower right',fontsize=8)
    ax2.set_xticks(np.arange(0, 4000, 250), minor=True)
    ax2.set_xticks(np.arange(0, 4000, 500))
    ax2.grid(True, color='gray', alpha=0.3, linestyle='dashed')
    ax2.set_ylabel('Magnitud [dB]')
    ax2.set_xlabel('Frecuencia [Hz]')

    ax3.plot(w_flujo_OVV, 10*np.log10(abs(h_flujo_OVV)), color='red', label='Flujo OVV',          linewidth=linewidth)
    ax3.plot(w_pb_pa,     10*np.log10(abs(h_pb_pa)),      color='k',alpha=0.75,   label='OVV filtrada PA-PB', linewidth=linewidth)
    ax3.legend(loc='lower right',fontsize=8)
    ax3.set_xticks(np.arange(0, 4000, 250), minor=True)
    ax3.set_xticks(np.arange(0, 4000, 500))
    ax3.grid(True, color='gray', alpha=0.3, linestyle='dashed')
    ax3.set_ylabel('Magnitud [dB]')
    ax3.set_xlabel('Frecuencia [Hz]')
    fig.canvas.draw()   # ← actualiza sin recrear ni parpadear


def graficar_filtro(change=None):
    global FF, BW1, flujo_ovv

    OVV_filtrada_pb_, b_pb_, a_pb_ = LP_Filter(OVV_original, fs=fs_r, fc=fc_filtro.value, order=orden_filtro.value)
    w_pb_, h_pb_                   = sig.freqz(b_pb_, a_pb_, fs=fs_r)
    OVV_filtrada_pb_pa_, _, _      = HP_Filter(OVV_filtrada_pb_, fs=fs_r)

    flujo_ovv, br, ar, FF, BW1 = IF_SNF_OneMetric(OVV_filtrada_pb_pa_, fs=fs_r, numMetric=metrica)

    slider.unobserve(graficar_slider, names='value')
    slider.value = FF
    slider.observe(graficar_slider, names='value')

    _dibujar(OVV_filtrada_pb_pa_, w_pb_, h_pb_, FF)

def graficar_slider(change):
    OVV_filtrada_pb_, b_pb_, a_pb_ = LP_Filter(OVV_original, fs=fs_r, fc=fc_filtro.value, order=orden_filtro.value)
    w_pb_, h_pb_                   = sig.freqz(b_pb_, a_pb_, fs=fs_r)
    OVV_filtrada_pb_pa_, _, _      = HP_Filter(OVV_filtrada_pb_, fs=fs_r)
    _dibujar(OVV_filtrada_pb_pa_, w_pb_, h_pb_, change['new'])

def reset_slider(b):
    slider.value = FF

def guardar(b):
    df = pd.read_csv(FC.selected, sep=',')
    df['flujo_OVV'] = flujo_ovv
    df.to_csv(FC.selected, index=False, sep=',', float_format='%.6f')
    print(f"Guardado: {Path(FC.selected).name}")

# --- Observers ---
orden_filtro.observe(graficar_filtro, names='value')
fc_filtro.observe(graficar_filtro,    names='value')
slider.observe(graficar_slider,       names='value')
reset_button.on_click(reset_slider)
guardar_button.on_click(guardar)

# --- Layout ---
grid_pb = widgets.GridBox(
    children=[orden_filtro, fc_filtro],
    layout=widgets.Layout(display='grid', grid_auto_flow='column',
                          grid_auto_columns='auto', grid_gap='0px 10px', width='50%')
)
grid_notch = widgets.GridBox(
    children=[slider, reset_button, guardar_button],
    layout=widgets.Layout(display='grid', grid_auto_flow='column',
                          grid_auto_columns='auto', grid_gap='0px 10px', width='80%')
)
titulo_filtro   = widgets.HTML(value="<b>Pasa bajos</b>")
titulo_notch    = widgets.HTML(value="<b>Notch</b>")
linea_divisoria = widgets.HTML(value="<hr style='border:1px solid #ccc; margin:10px 0;'>")

display(widgets.VBox([
    titulo_filtro, grid_pb, linea_divisoria,
    titulo_notch, grid_notch, linea_divisoria,
    fig_out
]))

graficar_filtro()